# ET Severity Training

Single- and multimodal 9-fold LOSO-CV training.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

from et_severity import (
    EncoderConfig,
    MultimodalModelConfig,
    SingleModelConfig,
    TrainingConfig,
    build_LOSO_loaders,
    build_manifest,
    run_loso_cv,
    set_seed,
)

In [ ]:
SEED = 42
DATA_ROOT = Path("/content/data") if Path("/content/data").exists() else REPO_ROOT / "data"
LABEL_CSV = DATA_ROOT / "relabel_md_k5.csv"

SEQ_LEN = 512
BATCH_SIZE = 16
TARGET_PER_CLASS = 200
NUM_WORKERS = 4

set_seed(SEED)
manifest = build_manifest(LABEL_CSV, DATA_ROOT, target_col="target_k5")
print("samples:", len(manifest))
print("patients:", sorted(manifest["patient_id"].unique()))

In [ ]:
training_config = TrainingConfig(
    epochs=100,
    learning_rate=1e-3,
    weight_decay=1e-4,
    optimizer="adam",
    monitor="loss",
    early_stopping_patience=20,
    grad_clip=1.0,
    use_scheduler=True,
    scheduler_patience=5,
)

ENCODER_PARAMS = {
    "LSTM": {"hidden_size": 128, "num_layers": 2, "dropout": 0.1},
    "ResNet18": {"feature_dim": 128},
    "TimesNet": {
        "feature_dim": 128,
        "d_ff": 128,
        "e_layers": 1,
        "top_k": 5,
        "dropout": 0.1,
    },
    "MyWaveNet": {
        "residual_channels": 128,
        "skip_channels": 128,
        "n_stacks": 2,
    },
}

def encoder_config(name):
    return EncoderConfig(name, dict(ENCODER_PARAMS[name]))

def single_config(name, num_classes):
    return SingleModelConfig(
        encoder=encoder_config(name),
        num_classes=num_classes,
        d_model=128,
        mil_attn_dim=64,
        seq_len=SEQ_LEN,
    )

## Single-modality LOSO-CV

In [ ]:
acc_loaders = build_LOSO_loaders(
    manifest,
    modality="acc",
    batch_size=BATCH_SIZE,
    target_per_class=TARGET_PER_CLASS,
    seg_len=SEQ_LEN,
    hop=SEQ_LEN,
    num_workers=NUM_WORKERS,
)

acc_result = run_loso_cv(
    acc_loaders,
    modality="acc",
    target="severity",
    model_config=single_config("LSTM", num_classes=4),
    training_config=training_config,
    seed=SEED,
    expected_folds=9,
    checkpoint_dir=REPO_ROOT / "checkpoints" / "acc_severity",
)
display(acc_result["fold_results"])
display(acc_result["fold_summary"])

## Multimodal LOSO-CV

In [ ]:
multimodal_loaders = build_LOSO_loaders(
    manifest,
    modality="multimodal",
    batch_size=BATCH_SIZE,
    target_per_class=TARGET_PER_CLASS,
    seg_len=SEQ_LEN,
    hop=SEQ_LEN,
    num_workers=NUM_WORKERS,
)

multimodal_config = MultimodalModelConfig(
    acc_encoder=encoder_config("LSTM"),
    traj_encoder=encoder_config("ResNet18"),
    num_classes=4,
    d_model=128,
    cross_attention_heads=8,
    mil_attn_dim=64,
    time_pool="attn",
    seq_len=SEQ_LEN,
)

multimodal_result = run_loso_cv(
    multimodal_loaders,
    modality="multimodal",
    model_config=multimodal_config,
    training_config=training_config,
    seed=SEED,
    expected_folds=9,
    checkpoint_dir=REPO_ROOT / "checkpoints" / "multimodal_severity",
)
display(multimodal_result["fold_results"])
display(multimodal_result["fold_summary"])